# Chapter 4: Word Representations

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch04_word_representations.ipynb)


## What is in this notebook, and what it needs before it runs

Three cells about word vectors, and all three reach outside the machine.

1. **Skip-gram with negative sampling, trained from scratch** on text8: roughly
   100 MB of cleaned Wikipedia, about 17 million tokens. The download is the
   short part and the training is the long part.
2. **Pre-trained vectors scored against human similarity judgements**, by
   Spearman correlation. The cell's own comment puts the downloads at roughly
   3 GB.
3. **PCA down to two dimensions** for nineteen words in four semantic
   categories, which is the picture of an embedding space the chapter argues
   from.

All three need `gensim`, and `gensim` is not installed on the machine this pack
is built on. That is why no cell below has stored output. To run it:
`pip install gensim`, then expect a download in the gigabytes and a training run
in cell 1 measured in tens of minutes on a laptop CPU.

**One thing to know before you start cell 2.** It loads
`glove-wiki-gigaword-50` three times, under the names `w2v`, `glove` and `ft`,
so the comparison it prints is one set of vectors against itself and the three
correlations are equal by construction rather than by result. Fixing that is the
exercise. Give `w2v` and `ft` genuinely different models,
`word2vec-google-news-300` and `fasttext-wiki-news-subwords-300`, and the
comparison starts to mean something. Those two are several more gigabytes, which
is better to know now than after the download starts.


> **This notebook was not executed when it was built, so no cell below has
> stored output.** The reason: all three cells need `gensim`, which is not installed on the machine this pack is built on, and cells 2 and 3 would then download roughly 3 GB of pre-trained vectors while cell 1 trains skip-gram over 17 million tokens.
>
> Nothing here is broken. It is code to read now and to run once you have what
> it needs, and the section above says what that is. Build it yourself with
> `python tools/build_notebook.py ch04` on a machine that has them.


### 4.3.5 Training in Practice

The gensim library provides the standard Python interface for training Word2Vec. The following example trains a Skip-gram model on a small corpus and explores the resulting embeddings:


In [ ]:
# Colab does not ship gensim. The smart_open bound is not version
# pinning for its own sake: gensim 4.x fails to import against
# smart_open 8, with 'cannot import name smart_open from smart_open'.
%pip install -q gensim "smart_open<7"


In [ ]:
import gensim.downloader as api
from gensim.models import Word2Vec

# Load text8: 100MB of cleaned, lowercased Wikipedia text (~17M tokens)
corpus = api.load("text8")

# Train Skip-gram with negative sampling
model = Word2Vec(
    sentences=corpus,
    vector_size=100,   # embedding dimension d
    window=5,          # context window c
    sg=1,              # 1 = Skip-gram, 0 = CBOW
    negative=5,        # number of negative samples K
    min_count=5,       # discard words with fewer occurrences
    epochs=5,          # training passes
    workers=4          # parallel threads
)

# Query most similar words (example output varies by seed and version)
print(model.wv.most_similar("king", topn=5))
# e.g. [('prince', 0.68), ('kings', 0.66), ('queen', 0.63), ...]

# Analogy: king - man + woman = ?
result = model.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
print(f"king - man + woman = {result[0][0]}")  # often 'queen' on a well-trained model

# Embedding shape
print(model.wv["king"].shape)  # (100,)


### 4.4.3 Comparing Embedding Methods

A reasonable question at this point: given Word2Vec, GloVe, and FastText, which should a practitioner choose? The following code compares all three on a small word similarity task:


In [ ]:
import gensim.downloader as api
from scipy.stats import spearmanr

# Load pre-trained embeddings.
# Note: these downloads total roughly 3 GB; expect a one-time delay.
w2v = api.load("glove-wiki-gigaword-50")
glove = api.load("glove-wiki-gigaword-50")
ft = api.load("glove-wiki-gigaword-50")

# Word pairs with human similarity scores (SimLex-999 subset)
pairs = [
    ("king", "queen", 8.5), ("cat", "dog", 5.0),
    ("happy", "sad", 2.3), ("fast", "slow", 1.9),
    ("computer", "laptop", 7.3)
]

print(f"{'Pair':<20} {'W2V':>6} {'GloVe':>6} {'FT':>6} {'Human':>6}")
print("-" * 48)
for w1, w2, human in pairs:
    s_w2v = w2v.similarity(w1, w2)
    s_glove = glove.similarity(w1, w2)
    s_ft = ft.similarity(w1, w2)
    print(f"{w1+'-'+w2:<20} {s_w2v:>6.3f} {s_glove:>6.3f} {s_ft:>6.3f} {human:>6.1f}")

# Spearman correlation with human judgments
human_scores = [h for _, _, h in pairs]
for name, model in [("W2V", w2v), ("GloVe", glove), ("FT", ft)]:
    model_scores = [model.similarity(w1, w2) for w1, w2, _ in pairs]
    rho, _ = spearmanr(model_scores, human_scores)
    print(f"{name} Spearman rho: {rho:.3f}")
# NOTE: Original chapter uses 'word2vec-google-news-300' (~1GB+). Substituted with 'glove-wiki-gigaword-50' (~66MB) for Colab.
# NOTE: Original chapter uses 'glove-wiki-gigaword-100' (~1GB+). Substituted with 'glove-wiki-gigaword-50' (~66MB) for Colab.
# NOTE: Original chapter uses 'fasttext-wiki-news-subwords-300' (~1GB+). Substituted with 'glove-wiki-gigaword-50' (~66MB) for Colab.


### 4.5.1 Intrinsic Evaluation: Analogy and Similarity

![Figure 4.6 -- Word analogy parallelogram](../figures/fig-04-6.pdf)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import gensim.downloader as api

# Load pre-trained GloVe vectors
glove = api.load("glove-wiki-gigaword-50")

# Select words from semantic categories
words = ["dog", "cat", "horse", "fish", "bird",
         "france", "germany", "italy", "spain", "japan",
         "red", "blue", "green", "yellow", "black",
         "king", "queen", "man", "woman"]

# Extract embedding matrix and reduce to 2D
vectors = np.array([glove[w] for w in words])
pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

# Plot with labels
plt.figure(figsize=(10, 8))
for i, word in enumerate(words):
    plt.scatter(coords[i, 0], coords[i, 1])
    plt.annotate(word, (coords[i, 0] + 0.02, coords[i, 1] + 0.02))
plt.title("GloVe Embeddings (PCA to 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.savefig("ch04_embedding_pca.pdf")  # illustrative only; the book figure is generated by figures/matplotlib/fig-04-3.py
print(f"Embedding shape: {vectors.shape}, PCA variance explained: {pca.explained_variance_ratio_.sum():.3f}")
plt.show()
# NOTE: Original chapter uses 'glove-wiki-gigaword-100' (~1GB+). Substituted with 'glove-wiki-gigaword-50' (~66MB) for Colab.


---

## Summary

This notebook demonstrated the key code examples from Chapter 4: Word Representations. For the full mathematical exposition and discussion, refer to the textbook chapter.
